# ElasticNet pool model
Global ElasticNet (L1+L2) trained on pooled data. Features: same as LSTM (return lags, volume_lag_1, RSI, MACD) + VIX 5-day SMA, **vix_velocity**, **vix_momentum**, month_sin/cos, Fear & Greed change. Multi-output: 21-step returns; **inputs scaled with StandardScaler before fit**.

In [32]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

LAG_RETURNS = 5
RSI_PERIOD = 14
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
# Load best alpha from random search if available (run random search cell first to generate)
import json
_ridge_path = ARTIFACTS_DIR / "best_ridge_params.json"
if _ridge_path.exists():
    with open(_ridge_path) as f:
        _ridge = json.load(f)
    RIDGE_ALPHA = _ridge["alpha"]
else:
    RIDGE_ALPHA = 1.0  # L2 regularization strength

In [33]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    """Features: LSTM-style (return lags, volume_lag_1, RSI, MACD) + VIX 5-day SMA, vix_velocity, vix_momentum, month sin/cos, Fear & Greed change. Target = next 21 returns. All scaled before fit."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        vix = df["vix"].astype(np.float64)
        df["vix_sma_5"] = vix.shift(1).rolling(5).mean()
        df["vix_velocity"] = vix.diff(1)
        df["vix_momentum"] = vix - df["vix_sma_5"]
    else:
        df["vix_sma_5"] = np.nan
        df["vix_velocity"] = np.nan
        df["vix_momentum"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    feature_cols_lstm = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + ["volume_lag_1", "rsi", "macd_line", "macd_signal"]
    feature_cols_xgb = ["vix_sma_5", "vix_velocity", "vix_momentum", "month_sin", "month_cos", "fear_greed_change"]
    feature_cols = feature_cols_lstm + feature_cols_xgb
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return"] + feature_cols + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols, target_cols


def train_global_elasticnet(stacked: pd.DataFrame, horizon: int):
    """Train one ElasticNet (MultiOutputRegressor) on pooled data. Returns dict for predict_elasticnet_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    X = pooled_feat[feature_cols].values.astype(np.float64)
    y = pooled_feat[target_cols].values.astype(np.float64)
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    ridge = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge.fit(X_s, y)
    return {"model": ridge, "scaler": scaler, "feature_cols": feature_cols}


def predict_elasticnet_global(context_df: pd.DataFrame, horizon: int, global_enet: dict) -> list:
    """Predict 21 price steps using pre-trained global Ridge."""
    if global_enet is None:
        return []
    try:
        feat_df, feature_cols, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < 1:
        return []
    X = feat_df[feature_cols].values.astype(np.float64)
    X_s = global_enet["scaler"].transform(X)
    last_row = X_s[-1:]
    pred_returns = global_enet["model"].predict(last_row).ravel()
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.cumprod(np.concatenate([[1.0], 1.0 + pred_returns]))[1:]
    return [float(p) for p in prices[:horizon]]

In [34]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-09,AAPL,121.089996,129525800,24.030001,43.360000
1,2021-03-10,AAPL,119.980003,111943300,22.559999,45.560000
2,2021-03-11,AAPL,121.959999,103026500,21.910000,50.480000
3,2021-03-12,AAPL,121.029999,88105100,20.690001,53.720000
4,2021-03-15,AAPL,123.989998,92403800,20.030001,56.520000
5,2021-03-16,AAPL,125.570000,115227900,19.790001,54.800000
6,2021-03-17,AAPL,124.760002,111932600,19.230000,57.866667
7,2021-03-18,AAPL,120.529999,121229700,21.580000,52.333333
8,2021-03-19,AAPL,119.989998,185549500,20.950001,50.833333
9,2021-03-22,AAPL,123.389999,111912300,18.879999,50.700000


In [35]:
# 20-iteration Random Search: Mean MAE across 21 horizons; save best params for stack notebooks
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer
import json

def mean_mae_multioutput(y_true, y_pred):
    """Mean MAE over all 21 horizon columns (higher is worse; we use neg_mean_mae for scoring)."""
    return np.mean(np.abs(y_true - y_pred))

neg_mean_mae_scorer = make_scorer(mean_mae_multioutput, greater_is_better=False)

pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
feat_dfs = []
for sym in pooled["symbol"].unique():
    grp = pooled[pooled["symbol"] == sym].copy()
    try:
        feat_df, feature_cols, target_cols = build_feature_df(grp)
    except Exception:
        continue
    if len(feat_df) < MIN_TRAIN_STACK + FORECAST_HORIZON:
        continue
    feat_dfs.append(feat_df)
pooled_feat = pd.concat(feat_dfs, ignore_index=True)
pooled_feat = pooled_feat.sort_values("timestamp").reset_index(drop=True)
X_rs = pooled_feat[feature_cols].values.astype(np.float64)
y_rs = pooled_feat[target_cols].values.astype(np.float64)

pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("multi", MultiOutputRegressor(Ridge(random_state=42)))
])
param_dist_ridge = {
    "multi__estimator__alpha": [1e-5, 5e-5, 1e-4, 5e-4, 0.001, 0.005, 0.01, 0.1, 1.0, 10.0],
}
tscv = TimeSeriesSplit(n_splits=5)
search_ridge = RandomizedSearchCV(
    pipe_ridge, param_dist_ridge, n_iter=20, cv=tscv, scoring=neg_mean_mae_scorer,
    random_state=42, n_jobs=-1, verbose=1
)
search_ridge.fit(X_rs, y_rs)

best_ridge = {k.replace("multi__estimator__", ""): float(v) for k, v in search_ridge.best_params_.items()}
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACTS_DIR / "best_ridge_params.json", "w") as f:
    json.dump(best_ridge, f, indent=2)
RIDGE_ALPHA = best_ridge["alpha"]
print("Best Mean MAE (CV):", -search_ridge.best_score_)
print("Best RIDGE_ALPHA:", RIDGE_ALPHA)
print("Saved to", ARTIFACTS_DIR / "best_ridge_params.json")

c:\capstone_project_unfc\env\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 10 is smaller than n_iter=20. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Mean MAE (CV): 0.013293633084840911
Best RIDGE_ALPHA: 10.0
Saved to C:\capstone_project_unfc\model\experiments-pool\artifacts\best_ridge_params.json


In [36]:
global_enet = train_global_elasticnet(stacked, FORECAST_HORIZON)
print("Global ElasticNet trained on", len(build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)), "pooled train rows.")

Global ElasticNet trained on 11960 pooled train rows.


In [37]:
# Ridge coefficients per horizon (each of 21 outputs has its own linear model)
if global_enet is not None:
    multi = global_enet["model"]
    cols = global_enet["feature_cols"]
    coefs = np.array([multi.estimators_[h].coef_ for h in range(len(multi.estimators_))])  # (21, n_features)
    intercepts = np.array([multi.estimators_[h].intercept_ for h in range(len(multi.estimators_))])
    df_coef = pd.DataFrame(coefs.T, index=cols, columns=[f"h{h+1}" for h in range(coefs.shape[0])])
    print("Ridge coefficients per horizon (rows=features, columns=horizon):")
    display(df_coef)
    mean_coef = df_coef.mean(axis=1)
    mean_abs_coef = df_coef.abs().mean(axis=1)
    summary = pd.DataFrame({"mean_coef": mean_coef, "mean_abs_coef": mean_abs_coef}).sort_values("mean_abs_coef", ascending=False)
    print("\nSummary (mean coefficient and mean |coefficient| across horizons):")
    display(summary)
    print("\nIntercepts per horizon:", intercepts)
else:
    print("No trained model (global_enet is None). Run the training cell above first.")

Ridge coefficients per horizon (rows=features, columns=horizon):


,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,...,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21
ret_lag_1,0.000014,-0.000513,-0.000023,-0.000018,0.000151,-0.000312,-0.000152,0.000542,-0.000230,0.000359,...,0.000624,-0.000429,-0.000116,0.000144,-0.000209,0.000540,-0.000265,0.000353,-0.000078,-0.000170
ret_lag_2,-0.000675,0.000028,0.000175,0.000278,-0.000253,-0.000207,0.000444,-0.000214,0.000307,-0.000046,...,-0.000335,-0.000266,0.000040,-0.000160,0.000578,-0.000213,0.000473,0.000027,-0.000302,-0.000175
ret_lag_3,-0.000053,0.000166,0.000320,-0.000143,-0.000186,0.000444,-0.000316,0.000344,-0.000067,0.000613,...,-0.000219,-0.000044,-0.000286,0.000599,-0.000217,0.000514,0.000128,-0.000146,-0.000240,-0.000154
ret_lag_4,0.000131,0.000236,-0.000130,-0.000082,0.000417,-0.000312,0.000298,-0.000043,0.000574,-0.000326,...,0.000054,-0.000353,0.000499,-0.000176,0.000482,0.000195,-0.000075,-0.000057,-0.000212,-0.000401
ret_lag_5,0.000264,-0.000165,-0.000144,0.000493,-0.000347,0.000319,-0.000054,0.000697,-0.000299,-0.000245,...,-0.000405,0.000423,-0.000298,0.000461,0.000121,-0.000074,-0.000112,-0.000040,-0.000369,-0.000453
volume_lag_1,0.000673,0.000444,0.000553,0.000536,0.000545,0.000488,0.000514,0.000534,0.000547,0.000499,...,0.000500,0.000281,0.000374,0.000472,0.000313,0.000382,0.000314,0.000354,0.000575,0.000509
rsi,0.000384,0.000122,0.000148,0.000295,0.000401,0.000290,-0.000182,-0.000445,-0.000141,-0.000300,...,-0.000208,0.000119,-0.000337,-0.000522,-0.000404,-0.000625,-0.000298,-0.000438,-0.000093,-0.000057
macd_line,-0.000952,-0.001009,-0.000970,-0.001245,-0.001096,-0.000874,-0.000520,-0.000920,-0.000349,-0.000058,...,0.001354,0.000745,0.000790,0.000704,0.000314,0.000776,0.000967,0.000885,0.000804,0.001053
macd_signal,0.000455,0.000702,0.000643,0.000761,0.000636,0.000465,0.000430,0.000940,0.000252,0.000123,...,-0.001283,-0.000933,-0.000726,-0.000635,-0.000302,-0.000624,-0.000985,-0.000783,-0.000826,-0.001105
vix_sma_5,0.000242,0.000171,0.000144,0.000060,0.000005,0.000037,0.000048,0.000105,0.000055,0.000097,...,-0.000056,-0.000066,-0.000120,-0.000175,-0.000162,-0.000201,-0.000271,-0.000180,-0.000238,-0.000244



Summary (mean coefficient and mean |coefficient| across horizons):


,mean_coef,mean_abs_coef
macd_line,0.000033,0.000794
macd_signal,-0.000147,0.000662
volume_lag_1,0.000465,0.000465
vix_momentum,0.000095,0.000363
vix_velocity,0.000011,0.000328
rsi,-0.000115,0.000282
ret_lag_2,0.000006,0.000277
ret_lag_5,-0.000012,0.000276
ret_lag_3,0.000034,0.000264
ret_lag_1,0.000006,0.000254



Intercepts per horizon: [0.00091469 0.00093909 0.00093611 0.0009165  0.00091752 0.00091978
 0.00091062 0.00090423 0.00089732 0.00090589 0.00091177 0.00090303
 0.00089553 0.00090272 0.00089451 0.00089283 0.00088828 0.00088527
 0.00088055 0.0008843  0.00087105]


In [38]:
model_name = "elasticnet"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_elasticnet_global(context_df, FORECAST_HORIZON, global_enet)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-10,278.779999,277.388943,0,AAPL
1,2025-12-11,278.029999,277.635606,0,AAPL
2,2025-12-12,278.279999,277.820489,0,AAPL
3,2025-12-15,274.109985,277.845365,0,AAPL
4,2025-12-16,274.609985,277.911873,0,AAPL


In [39]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_elasticnet_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_elasticnet_pool.parquet")

         model   symbol        MAE       RMSE    MAPE_%
0   elasticnet     AAPL  10.788102  12.972320  4.099653
1   elasticnet     MSFT  27.614160  31.869450  6.431914
2   elasticnet    GOOGL  13.426785  15.371794  4.203751
3   elasticnet     AMZN  13.327648  15.721461  6.125228
4   elasticnet      JPM  13.406035  14.799315  4.268150
5   elasticnet      JNJ  11.689273  13.036817  5.092406
6   elasticnet      WMT   5.109318   5.825750  4.156777
7   elasticnet      SPY   5.688651   7.043377  0.829896
8   elasticnet      XOM   7.706976   9.080816  5.506322
9   elasticnet     NVDA   5.363678   6.845178  2.937339
10  elasticnet  overall  11.412063  15.759329  4.365144
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_elasticnet_pool.parquet
